# Social Network Graph Demo

This notebook demonstrates a **social network** modelled as a property graph and queried
using Gremlin — backed by either **H2** (embedded, zero-config) or **Apache Iceberg via Trino**
(lakehouse, requires Docker).

<img src="image/SocialNetworkingGraphRepresentation.png" alt="Social Network Graph" style="max-width:100%; height:auto; display:block; margin:1em 0;">

## Graph schema

```
Person ──KNOWS──► Person
Person ──WORKS_AT──► Company
Person ──LIVES_IN──► City
Person ──HAS_SKILL──► Skill
Company ──LOCATED_IN──► City
```

<div style="display:flex; gap:2em; text-align:left">

<div>

| Vertex | Key properties |
|:-------|:---------------|
| `Person` | name, age, email, city, country |
| `Company` | name, industry, country, founded, employees |
| `City` | name, country, population, timezone |
| `Skill` | name, category |

</div>

<div>

| Edge | Key properties |
|:-----|:---------------|
| `KNOWS` | since, strength (strong/moderate/weak) |
| `WORKS_AT` | role, startYear |
| `LIVES_IN` | since |
| `HAS_SKILL` | level (Expert/Intermediate/Beginner), yearsOfExp |
| `LOCATED_IN` | — |

</div>

</div>

---
## Step 0 — Setup

In [1]:
import requests, pandas as pd, json as _json, subprocess, pathlib, sys, time, os
from IPython.display import display, Markdown

# ── Configuration ──────────────────────────────────────────────────────────────
# NOTEBOOK_BACKEND env var controls the backend when running in Docker.
# docker compose up engine jupyter  -> NOTEBOOK_BACKEND=h2   (H2 only)
# docker compose up                 -> NOTEBOOK_BACKEND=iceberg (full stack)
import os as _os_be
BACKEND   = _os_be.environ.get('NOTEBOOK_BACKEND', 'iceberg')  # 'h2'  'iceberg'
import os as _os
BASE_URL  = _os.environ.get('BASE_URL', 'http://localhost:7000')
SHOW_PLAN = True

def _root():
    import os as _os_rr
    if 'REPO_ROOT' in _os_rr.environ:
        return pathlib.Path(_os_rr.environ['REPO_ROOT'])
    for p in [pathlib.Path.cwd()] + list(pathlib.Path.cwd().parents):
        if (p / 'pom.xml').exists(): return p
    return pathlib.Path.cwd()

ROOT        = _root()
MAPPING_DIR = ROOT / 'demo' / 'social_networking' / 'mappings'

import os as _os_mp
# ENGINE_TRINO_HOST is the hostname the *engine* (inside Docker) uses to reach Trino.
# ENGINE_TRINO_HOST: hostname the *engine* uses to reach Trino.
# Defaults to 'localhost' for local runs; set ENGINE_TRINO_HOST=trino when
# the engine runs inside Docker (where the Trino service name is 'trino').
_engine_trino_host = _os_mp.environ.get('ENGINE_TRINO_HOST', 'localhost')
_engine_trino_port = _os_mp.environ.get('ENGINE_TRINO_PORT', '8080')
# TRINO_HOST / TRINO_PORT are used for direct notebook→Trino connections (seed script etc.)
_trino_host = _os_mp.environ.get('TRINO_HOST', 'localhost')
_trino_port = _os_mp.environ.get('TRINO_PORT', '8080')

def _build_iceberg_mapping():
    """Build the social-network iceberg mapping with the correct Trino URL."""
    return {
        'backends': {'iceberg': {
            'url': f'jdbc:trino://{_engine_trino_host}:{_engine_trino_port}/iceberg',
            'user': 'admin', 'password': '', 'driverClass': 'io.trino.jdbc.TrinoDriver'
        }},
        'defaultDatasource': 'iceberg',
        'vertices': {
            'Person':  {'table': 'iceberg:iceberg.social_network.persons',  'idColumn': 'id',
                        'properties': {'name': 'name', 'age': 'age', 'email': 'email',
                                       'city': 'city', 'country': 'country', 'joinedAt': 'joined_at'}},
            'Company': {'table': 'iceberg:iceberg.social_network.companies', 'idColumn': 'id',
                        'properties': {'name': 'name', 'industry': 'industry', 'country': 'country',
                                       'founded': 'founded', 'employees': 'employees'}},
            'City':    {'table': 'iceberg:iceberg.social_network.cities',    'idColumn': 'id',
                        'properties': {'name': 'name', 'country': 'country',
                                       'population': 'population', 'timezone': 'timezone'}},
            'Skill':   {'table': 'iceberg:iceberg.social_network.skills',    'idColumn': 'id',
                        'properties': {'name': 'name', 'category': 'category'}},
        },
        'edges': {
            'KNOWS':     {'table': 'iceberg:iceberg.social_network.knows',       'idColumn': 'id',
                          'outColumn': 'person_a_id', 'inColumn': 'person_b_id',
                          'outVertexLabel': 'Person',  'inVertexLabel': 'Person',
                          'properties': {'since': 'since', 'strength': 'strength'}},
            'WORKS_AT':  {'table': 'iceberg:iceberg.social_network.works_at',    'idColumn': 'id',
                          'outColumn': 'person_id',   'inColumn': 'company_id',
                          'outVertexLabel': 'Person',  'inVertexLabel': 'Company',
                          'properties': {'role': 'role', 'startYear': 'start_year'}},
            'LIVES_IN':  {'table': 'iceberg:iceberg.social_network.lives_in',    'idColumn': 'id',
                          'outColumn': 'person_id',   'inColumn': 'city_id',
                          'outVertexLabel': 'Person',  'inVertexLabel': 'City',
                          'properties': {'since': 'since'}},
            'HAS_SKILL': {'table': 'iceberg:iceberg.social_network.has_skill',   'idColumn': 'id',
                          'outColumn': 'person_id',   'inColumn': 'skill_id',
                          'outVertexLabel': 'Person',  'inVertexLabel': 'Skill',
                          'properties': {'level': 'level', 'yearsOfExp': 'years_of_exp'}},
            'LOCATED_IN':{'table': 'iceberg:iceberg.social_network.company_city','idColumn': 'id',
                          'outColumn': 'company_id',  'inColumn': 'city_id',
                          'outVertexLabel': 'Company', 'inVertexLabel': 'City',
                          'properties': {}},
        },
    }

# Mapping IDs per backend
MAPPING_FILES = {
    'h2':      MAPPING_DIR / 'social-network-h2-mapping.json',
    'iceberg': None,  # built dynamically — see _build_iceberg_mapping()
}
MAPPING_IDS = {
    'h2':      'social-h2',
    'iceberg': 'social-iceberg',
}

QUERY_TIMEOUT = 120  # seconds

# ── Query helpers ──────────────────────────────────────────────────────────────
def gremlin(q, timeout=None):
    if timeout is None: timeout = QUERY_TIMEOUT
    r = requests.post(f'{BASE_URL}/gremlin/query', json={'gremlin': q}, timeout=timeout)
    d = r.json()
    if 'error' in d:
        display(Markdown(f'\u26a0\ufe0f Query error: `{d["error"]}`'))
    return d

def show(q, title='', limit=20):
    if title: display(Markdown(f'### {title}'))
    display(Markdown(f'**Gremlin:** `{q}`'))
    res  = gremlin(q)
    rows = res.get('results', [])
    display(Markdown(f'**Results:** {res.get("resultCount", len(rows))}'))
    if rows:
        if isinstance(rows[0], dict): display(pd.DataFrame(rows[:limit]))
        else:
            for i, v in enumerate(rows[:limit], 1): print(f'{i}. {v}')

# ── Engine startup ─────────────────────────────────────────────────────────────
def _start_engine_if_needed() -> bool:
    try:
        return requests.get(f'{BASE_URL}/health', timeout=5).ok
    except Exception:
        pass
    display(Markdown('\u23f3 Engine not running \u2014 starting with `mvn exec:java`\u2026'))
    subprocess.Popen(['mvn', 'exec:java'], cwd=str(ROOT),
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(40):
        time.sleep(3)
        try:
            if requests.get(f'{BASE_URL}/health', timeout=3).ok:
                return True
        except Exception:
            pass
    return False

if not _start_engine_if_needed():
    display(Markdown('\u274c Engine did not start. Run `mvn exec:java` in a terminal and re-run this cell.'))
else:
    h = requests.get(f'{BASE_URL}/health', timeout=5).json()
    backends = [b['id'] for b in h.get('backends', [])]
    display(Markdown(
        f'\u2705 Engine `{h["status"]}` \u2014 provider: **`{h.get("provider","?")}`**'
        f' \u2014 backends: `{backends}`\n\n'
        f'> \u2139\ufe0f Backend connections are declared in the mapping file and auto-registered when the mapping is uploaded (Step 1).'
    ))


✅ Engine `ok` — provider: **`sql-multi`** — backends: `['default']`

> ℹ️ Backend connections are declared in the mapping file and auto-registered when the mapping is uploaded (Step 1).

---
## Step 1 — Upload Mapping & Seed Data

Uploads the mapping for the chosen backend and seeds the sample graph
(10 people, 5 companies, 10 cities, 8 skills and all relationship edges).

> ⚠️ **Iceberg mode:** Start the Docker stack before running this cell:
> ```bash
> bash /path/to/demo/infra/scripts/iceberg_local_up.sh
> ```

| Flag | Behaviour |
|------|-----------|
| `WIPE = False` | *(default)* Skip seed if data already present |
| `WIPE = True` | Drop all existing data and reload from scratch |

In [2]:
WIPE = False
_ok_to_proceed = True

# ── Iceberg container guard ────────────────────────────────────────────────────
if BACKEND == 'iceberg':
    ICEBERG_UP_SH = str(ROOT / 'demo' / 'infra' / 'scripts' / 'iceberg_local_up.sh')

    def _containers_running():
        import urllib.request
        for host in ('trino', 'localhost'):
            try:
                urllib.request.urlopen(f'http://{host}:8080/v1/info', timeout=5)
                return True
            except Exception:
                continue
        return False

    if not _containers_running():
        _ok_to_proceed = False
        display(Markdown(
            '\u274c **Iceberg Docker containers are not running.**\n\n'
            'Start them in a terminal first, then re-run this cell:\n'
            '```bash\n'
            f'bash {ICEBERG_UP_SH}\n'
            '```'
        ))
    else:
        display(Markdown('\u2705 Iceberg containers are running'))

# ── Upload mapping ─────────────────────────────────────────────────────────────
if _ok_to_proceed:
    mapping_id = MAPPING_IDS[BACKEND]

    if BACKEND == 'iceberg':
        # Build mapping dynamically so the Trino URL always matches TRINO_HOST/TRINO_PORT.
        # Works whether the notebook runs in Docker Jupyter (TRINO_HOST=trino)
        # or locally against a Docker-hosted engine (TRINO_HOST defaults to localhost).
        import json as _json_mp
        _payload = _json_mp.dumps(_build_iceberg_mapping()).encode('utf-8')
        r = requests.post(f'{BASE_URL}/mapping/upload',
                          files={'file': ('social-iceberg.json', _payload, 'application/json')},
                          params={'id': mapping_id, 'name': f'Social {BACKEND.title()}', 'activate': 'true'},
                          timeout=15)
    else:
        mapping_file = MAPPING_FILES[BACKEND]
        with open(mapping_file, 'rb') as f:
            r = requests.post(f'{BASE_URL}/mapping/upload',
                              files={'file': (mapping_file.name, f, 'application/json')},
                              params={'id': mapping_id, 'name': f'Social {BACKEND.title()}', 'activate': 'true'},
                              timeout=15)
    if r.ok:
        d = r.json()
        msg = f'\u2705 Mapping uploaded \u2014 `{d["mappingId"]}`'
        if d.get('autoRegisteredBackends'):
            msg += f'  *(auto-registered backends: `{d["autoRegisteredBackends"]}`)*'
        display(Markdown(msg))
    else:
        _ok_to_proceed = False
        display(Markdown(f'\u274c Mapping upload failed: {r.text}'))

# ── Seed data ──────────────────────────────────────────────────────────────────
if _ok_to_proceed:
    seed_script = ROOT / 'demo' / 'social_networking' / 'scripts' / 'seed_social_network.py'
    cmd = [sys.executable, str(seed_script), '--backend', BACKEND,
           '--h2-mode', 'csvread', '--iceberg-mode', 'csv']
    if WIPE:
        cmd.append('--wipe')
    display(Markdown(
        f'Running: `{" ".join(cmd)}`\n'
        f'> **Load method:** {"H2 CSVREAD() bulk insert" if BACKEND == "h2" else "MinIO CSV upload + Hive external table INSERT SELECT"}'
    ))
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        display(Markdown(f'\u274c Seed failed:\n```\n{result.stderr[-800:]}\n```'))
    else:
        display(Markdown('\u2705 **Seed complete!**'))


✅ Mapping uploaded — `social-h2`

Running: `/Users/vjoshi/.pyenv/versions/3.13.9/bin/python3.13 /Users/vjoshi/SourceCode/graph-query-engine/demo/social_networking/scripts/seed_social_network.py --backend h2 --h2-mode csvread --iceberg-mode csv`
> **Load method:** H2 CSVREAD() bulk insert

Mapping uploaded: mapping-1776402123647
H2 already contains 10 person(s) — skipping seed. Use --wipe to reload.



✅ **Seed complete!**

---
## Q1 — All People

List every `Person` vertex with name, age and city.

In [4]:
show("g.V().hasLabel('Person').valueMap('name','age','city').limit(10)", title='Q1 — All people')

### Q1 — All people

**Gremlin:** `g.V().hasLabel('Person').valueMap('name','age','city').limit(10)`

**Results:** 10

,name,age,city
0,[Alice],[29],[San Francisco]
1,[Bob],[34],[New York]
2,[Carol],[27],[London]
3,[Dave],[41],[Berlin]
4,[Eve],[25],[Paris]
5,[Frank],[38],[Tokyo]
6,[Grace],[32],[Sydney]
7,[Hank],[45],[Toronto]
8,[Iris],[30],[Singapore]
9,[Jake],[22],[Amsterdam]


---
## Q2 — Find a Person by Name

Point lookup on a property — equivalent to SQL `WHERE name = 'Alice'`.

In [9]:
show("g.V().hasLabel('Person').has('name','Alice').valueMap('name','age','email','city')", title='Q2 — Find Alice')

### Q2 — Find Alice

**Gremlin:** `g.V().hasLabel('Person').has('name','Alice').valueMap('name','age','email','city')`

**Results:** 1

,name,age,email,city
0,[Alice],[29],[alice@example.com],[San Francisco]


---
## Q3 — Direct Friends (1-hop KNOWS)

Follow one `KNOWS` edge to find Alice's immediate connections.

In [5]:
show("g.V().hasLabel('Person').has('name','Alice').out('KNOWS').values('name')", title="Q3 — Alice's direct friends")

### Q3 — Alice's direct friends

**Gremlin:** `g.V().hasLabel('Person').has('name','Alice').out('KNOWS').values('name')`

**Results:** 3

1. Bob
2. Carol
3. Iris


---
## Q4 — Friends-of-Friends (2-hop)

Two consecutive `KNOWS` hops — the classic **Six Degrees of Separation** pattern.

In [6]:
show("g.V().hasLabel('Person').has('name','Alice').out('KNOWS').out('KNOWS').values('name').dedup()", title='Q4 — Friends-of-friends of Alice')

### Q4 — Friends-of-friends of Alice

**Gremlin:** `g.V().hasLabel('Person').has('name','Alice').out('KNOWS').out('KNOWS').values('name').dedup()`

**Results:** 2

1. Eve
2. Dave


---
## Q5 — Where Does Someone Work?

`Person → WORKS_AT → Company`

In [7]:
show("g.V().hasLabel('Person').has('name','Bob').out('WORKS_AT').valueMap('name','industry','country')", title='Q5 — Where does Bob work?')

### Q5 — Where does Bob work?

**Gremlin:** `g.V().hasLabel('Person').has('name','Bob').out('WORKS_AT').valueMap('name','industry','country')`

**Results:** 1

,name,industry,country
0,[FinanceHub],[Finance],[UK]


---
## Q6 — Colleagues (Person → Company → Person)

Bi-directional traversal: out to the company, then in to find co-workers.

In [8]:
show("g.V().hasLabel('Person').has('name','Alice').out('WORKS_AT').in('WORKS_AT').values('name').dedup()", title="Q6 — Alice's colleagues")

### Q6 — Alice's colleagues

**Gremlin:** `g.V().hasLabel('Person').has('name','Alice').out('WORKS_AT').in('WORKS_AT').values('name').dedup()`

**Results:** 2

1. Alice
2. Dave


---
## Q7 — Full Path: Person → Company → City

`path().by('name')` captures the complete traversal chain across three entity types.

In [9]:
show("g.V().hasLabel('Person').out('WORKS_AT').out('LOCATED_IN').path().by('name').limit(10)", title='Q7 — Person -> Company -> City path')

### Q7 — Person -> Company -> City path

**Gremlin:** `g.V().hasLabel('Person').out('WORKS_AT').out('LOCATED_IN').path().by('name').limit(10)`

**Results:** 10

1. ['Dave', 'TechCorp', 'San Francisco']
2. ['Alice', 'TechCorp', 'San Francisco']
3. ['Grace', 'FinanceHub', 'London']
4. ['Bob', 'FinanceHub', 'London']
5. ['Hank', 'DataStream', 'Berlin']
6. ['Carol', 'DataStream', 'Berlin']
7. ['Iris', 'CloudNine', 'New York']
8. ['Eve', 'CloudNine', 'New York']
9. ['Jake', 'AIWorks', 'Singapore']
10. ['Frank', 'AIWorks', 'Singapore']


---
## Q8 — Filter on Edge Property

The `strength` predicate lives on the `KNOWS` edge.

In [10]:
show("g.V().hasLabel('Person').has('name','Alice').outE('KNOWS').has('strength','strong').inV().values('name')", title="Q8 — Alice's strong connections")

### Q8 — Alice's strong connections

**Gremlin:** `g.V().hasLabel('Person').has('name','Alice').outE('KNOWS').has('strength','strong').inV().values('name')`

**Results:** 1

1. Bob


---
## Q9 — Reverse Lookup: Who Knows Python?

Start at a `Skill` vertex and walk backwards via `in('HAS_SKILL')`.

In [11]:
show("g.V().hasLabel('Skill').has('name','Python').in('HAS_SKILL').values('name')", title='Q9 — Who knows Python?')

### Q9 — Who knows Python?

**Gremlin:** `g.V().hasLabel('Skill').has('name','Python').in('HAS_SKILL').values('name')`

**Results:** 3

1. Alice
2. Carol
3. Jake


---
## Q10 — Expert-Level Skill Holders

`WHERE EXISTS` existence check on the `HAS_SKILL` edge.

In [12]:
show("g.V().hasLabel('Person').where(outE('HAS_SKILL').has('level','Expert')).values('name')", title='Q10 — Expert-level skill holders')

### Q10 — Expert-level skill holders

**Gremlin:** `g.V().hasLabel('Person').where(outE('HAS_SKILL').has('level','Expert')).values('name')`

**Results:** 7

1. Alice
2. Bob
3. Carol
4. Dave
5. Frank
6. Hank
7. Iris


---
## Q11 — Degree Centrality

Count `KNOWS` edges per person to find the most connected hub.

In [13]:
display(Markdown('### Q12 — Degree Centrality'))
people = gremlin("g.V().hasLabel('Person').values('name')").get('results', [])
if not people:
    display(Markdown('⚠️ No people found — run Step 1 (seed) first.'))
else:
    rows = []
    for name in people:
        out_d = int((gremlin(f"g.V().hasLabel('Person').has('name','{name}').out('KNOWS').count()").get('results') or [0])[0] or 0)
        in_d  = int((gremlin(f"g.V().hasLabel('Person').has('name','{name}').in('KNOWS').count()").get('results')  or [0])[0] or 0)
        rows.append({'Person': name, 'Out-degree': out_d, 'In-degree': in_d, 'Total': out_d + in_d})
    display(pd.DataFrame(rows).sort_values('Total', ascending=False).reset_index(drop=True))

### Q12 — Degree Centrality

,Person,Out-degree,In-degree,Total
0,Alice,3,0,3
1,Bob,1,1,2
2,Carol,1,1,2
3,Dave,1,1,2
4,Eve,1,1,2
5,Frank,1,1,2
6,Grace,1,1,2
7,Hank,1,1,2
8,Iris,0,2,2
9,Jake,0,1,1


---
## Q12 — Companies by Industry

List all companies with their industry and country.

In [14]:
show("g.V().hasLabel('Company').valueMap('name','industry','country','employees')", title='Q13 — Companies by Industry')

### Q13 — Companies by Industry

**Gremlin:** `g.V().hasLabel('Company').valueMap('name','industry','country','employees')`

**Results:** 5

,name,industry,country,employees
0,[TechCorp],[Technology],[USA],[5000]
1,[FinanceHub],[Finance],[UK],[2500]
2,[DataStream],[Data Analytics],[DE],[800]
3,[CloudNine],[Cloud Services],[USA],[3200]
4,[AIWorks],[AI/ML],[SG],[450]


---
## Q13 — Cities with Large Population

Filter cities by population — pushed down as a SQL `WHERE` clause.

In [15]:
show("g.V().hasLabel('City').has('population', gt(3000000)).valueMap('name','country','population')", title='Q14 — Large Cities (population > 3 million)')

### Q14 — Large Cities (population > 3 million)

**Gremlin:** `g.V().hasLabel('City').has('population', gt(3000000)).valueMap('name','country','population')`

**Results:** 6

,name,country,population
0,[New York],[USA],[8336817]
1,[London],[UK],[8982000]
2,[Berlin],[DE],[3769495]
3,[Tokyo],[JP],[13960000]
4,[Sydney],[AU],[5312000]
5,[Singapore],[SG],[5850000]


---
## Q14 — Person's City Timezone

`Person → LIVES_IN → City` — a two-hop traversal to look up where each person lives
and that city's timezone.

In [16]:
display(Markdown('### Q15 — Person City Timezone'))
display(Markdown(f'**Gremlin:** `g.V().hasLabel("Person").has("name","Alice").out("LIVES_IN").valueMap(...)`'))

people_sample = ['Alice', 'Bob', 'Carol', 'Dave', 'Eve']
rows_out = []
for person in people_sample:
    city_res = gremlin(f"g.V().hasLabel('Person').has('name','{person}').out('LIVES_IN').valueMap('name','country','timezone','population')")
    c_rows = city_res.get('results', [])
    if c_rows:
        c = c_rows[0]
        def _v(k): return (c.get(k) or ['?'])[0] if isinstance(c.get(k), list) else c.get(k, '?')
        rows_out.append({'Person': person, 'City': _v('name'), 'Country': _v('country'),
                         'Timezone': _v('timezone'), 'Population': _v('population')})

display(Markdown(f'**Results:** {len(rows_out)}'))
if rows_out: display(pd.DataFrame(rows_out))

### Q15 — Person City Timezone

**Gremlin:** `g.V().hasLabel("Person").has("name","Alice").out("LIVES_IN").valueMap(...)`

**Results:** 5

,Person,City,Country,Timezone,Population
0,Alice,San Francisco,USA,America/Los_Angeles,883305
1,Bob,New York,USA,America/New_York,8336817
2,Carol,London,UK,Europe/London,8982000
3,Dave,Berlin,DE,Europe/Berlin,3769495
4,Eve,Paris,FR,Europe/Paris,2161000


---
## Q15 — Skill Popularity

For each skill, count how many people hold it and how many are experts.

In [17]:
display(Markdown('### Q16 — Skill Popularity'))
skill_res  = gremlin("g.V().hasLabel('Skill').valueMap('name','category')")
skill_rows = skill_res.get('results', [])

report = []
for s in skill_rows:
    def _v(k): return (s.get(k) or ['?'])[0] if isinstance(s.get(k), list) else s.get(k, '?')
    sname    = _v('name')
    category = _v('category')
    count   = int((gremlin(f"g.V().hasLabel('Skill').has('name','{sname}').in('HAS_SKILL').count()").get('results') or [0])[0] or 0)
    experts = int((gremlin(f"g.V().hasLabel('Skill').has('name','{sname}').in('HAS_SKILL').where(outE('HAS_SKILL').has('level','Expert')).count()").get('results') or [0])[0] or 0)
    report.append({'Skill': sname, 'Category': category, 'Holders': count, 'Experts': experts})

if report:
    display(pd.DataFrame(report).sort_values('Holders', ascending=False).reset_index(drop=True))

### Q16 — Skill Popularity

,Skill,Category,Holders,Experts
0,Python,Programming,3,3
1,Machine Learning,AI/ML,3,3
2,SQL,Data,2,2
3,Kubernetes,DevOps,2,0
4,React,Frontend,2,2
5,Java,Programming,1,1
6,Graph Databases,Data,1,0
7,Spark,Data Engineering,1,1


---
## Q16 — Company Headquarters City

`Company → LOCATED_IN → City` — follow the edge to find where each company is based.

In [18]:
display(Markdown(f'**Gremlin:** `g.V().hasLabel("Company").out("LOCATED_IN").path().by("name").limit(10)`'))
res  = gremlin("g.V().hasLabel('Company').out('LOCATED_IN').path().by('name').limit(10)")
rows = res.get('results', [])
display(Markdown(f'**Results:** {res.get("resultCount", len(rows))}'))
for r in rows[:10]:
    if isinstance(r, list):
        display(Markdown(f'- {" -> ".join(str(x) for x in r)}'))
    else:
        print(r)

**Gremlin:** `g.V().hasLabel("Company").out("LOCATED_IN").path().by("name").limit(10)`

**Results:** 5

- TechCorp -> San Francisco

- FinanceHub -> London

- DataStream -> Berlin

- CloudNine -> New York

- AIWorks -> Singapore

---
## Backend Comparison

The **same Gremlin queries** work against both backends — only the mapping and
connection change.

| Feature | H2 (embedded) | Iceberg / Trino |
|---------|---------------|-----------------|
| Setup | Zero-config | Docker stack required |
| Storage | Local H2 file | MinIO / S3 (Parquet) |
| SQL dialect | H2 SQL | Trino / Iceberg SQL |
| Scale | Dev / testing | Production |

> ℹ️ **Datasource routing** — The engine inspects the `datasource` field on each
> vertex/edge mapping entry and routes the generated SQL to the matching registered
> backend (`BackendRegistry`). A single Gremlin query always targets one backend;
> cross-database joins are not supported within a single traversal.

---
## Done

Switch `BACKEND = 'h2'` or `'iceberg'` in Step 0 and re-run from the top.